In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

folder = Path("2025-citybike-data")
csv_files = sorted(folder.glob("*.csv"))
df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

print(f"Brakujące dane:\n{df.isna().sum()}")
df = df.dropna()
print(f"Liczba pełnych rekordów: {len(df)}")

In [ ]:
# obliczenie czasu przejazdu [h]
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])
df["time"] = (df["ended_at"] - df["started_at"]).dt.total_seconds()/3600

# odrzucamy błędne dane gdzie czas oddania jest wcześniejszy niż wypożyczenia
# odrzucamy błędne dane gdzie czas wypożyczenia jest dłuższy niż 1 dzień
df = df[(df["time"] > 0) & (df["time"] < 24)]

In [ ]:
df["hour"] = df["started_at"].dt.hour
df["weekday"] = df["started_at"].dt.weekday
df["month"] = df["started_at"].dt.month

monthly_counts = (df.groupby(["month", "rideable_type"]).size().reset_index(name="rides"))
pivot = monthly_counts.pivot(index="month", columns="rideable_type", values="rides")
pivot.plot(kind="line", marker="o",figsize=(10,6))

plt.xlabel("Miesiąc")
plt.ylabel("Liczba przejazdów")
plt.title("Liczba przejazdów według miesiąca i typu roweru")
plt.xticks(range(1,13))
plt.grid(True)
plt.savefig("miesiace.png")
plt.show()

In [ ]:
# obliczenie przejechanego dystansu [km]
def dist(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df["distance"] = dist(df["start_lat"], df["start_lng"], df["end_lat"], df["end_lng"])

# odrzucamy dane gdzie stacja początkowa i końcowa są takie same
df = df[df["distance"]>0]

In [ ]:
# prędkość w km/h
df["velocity"] = df["distance"]/df["time"]

In [ ]:
classic = df[df["rideable_type"] == "classic_bike"]
electric = df[df["rideable_type"] == "electric_bike"]

print(f"Liczba wypożyczeń rowerów klasycznych: {len(classic)}")
print(f"Liczba wypożyczeń rowerów elektrycznych: {len(electric)}")

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(electric["velocity"], bins=30, alpha=0.4, label="Elektryczne")
sns.histplot(classic["velocity"], bins=30, alpha=0.6, label="Klasyczne")

electric_mean = electric["velocity"].mean()
classic_mean = classic["velocity"].mean()

plt.axvline(electric_mean, linestyle="--", linewidth=2,
            label=f"Średnia dla rowerów elektrycznych: {electric_mean:.2f} km/h")
plt.axvline(classic_mean, color = "orange", linestyle="--", linewidth=2,
            label=f"Średnia dla rowerów klasycznych: {classic_mean:.2f} km/h")

plt.xlabel("Prędkość [km/h]")
plt.ylabel("Liczba")
plt.title("Rozkład prędkości rowerów")
plt.legend()
plt.savefig("predkosci.png")
plt.show()

In [ ]:
#chodziłam na zajęcia  z uczenia maszynowego stąd próba zabawy z drzewami decyzyjnymi:)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

In [ ]:
dzielnice = {
    "Manhattan": (40.7831, -73.9712),
    "Brooklyn": (40.6782, -73.9442),
    "Queens": (40.7282, -73.7949),
    "Bronx": (40.8448, -73.8648),
    "Staten Island": (40.5795, -74.1502),
}

def znajdz_dzielnice(lat, lng):
    return min(dzielnice.keys(), key=lambda b: dist(lat, lng, dzielnice[b][0], dzielnice[b][1]))

df["start_borough"] = df.apply(lambda row: znajdz_dzielnice(row["start_lat"], row["start_lng"]), axis=1)

In [ ]:
df["time_of_day"] = np.select([
    (df.hour >= 6) & (df.hour < 12),
    (df.hour >= 12) & (df.hour < 18),
    (df.hour >= 18)],
    [1, 2, 3],
    default=4
)

df["season"] = np.select([
    df["month"].isin([3,4,5]),
    df["month"].isin([6,7,8]),
    df["month"].isin([9,10,11]),
    df["month"].isin([1,2,12])],
    [1,2,3,4]
)

df["distance_bin"] = pd.cut(
    df["distance"],
    bins=[0, 0.6, 0.8, 1, 1.25, 1.5, 2, np.inf],
    labels=[1, 2, 3, 4, 5, 6, 7]
).astype(int)

dummies = pd.get_dummies(df["start_borough"], prefix="is")
df = pd.concat([df, dummies], axis=1)

df["member_casual"] = np.where(df.member_casual == "member", 1, 0)

print(df.columns)

In [ ]:
features = ['member_casual', 'time_of_day', 'season','distance_bin', 'is_Brooklyn', 'is_Manhattan']
X = df[features]
y = (df["rideable_type"] == "electric_bike").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

model = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                               max_depth=15, min_samples_split=10, min_samples_leaf=5)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print(f"ACCURACY: {metrics.accuracy_score(y_test, y_pred):.4f}")
print(f"MCC: {metrics.matthews_corrcoef(y_test, y_pred):.4f}")
cm = metrics.confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, cmap="YlOrRd")
plt.title("Macierz pomyłek")
plt.xlabel("Przewidywane")
plt.ylabel("Prawdziwe")
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(data=importance, x="importance", y="feature", palette="pastel")
plt.xlabel("Istotność")
plt.ylabel("Czynnik")
plt.savefig("importance.png")
plt.show()

Wyniki pokazują że model prawie całkowicie opiera się na dystansie pomiędzy stacjami, inne czynniki nie grają tutaj dużej roli.
Otrzymano accuracy bliskie 70% jednak nie jest to dobra metryka w tym przypadku ponieważ klasy nie są zbalansowane. 
Dużo lepszą metryką będzie MCC który wyniósł około 0.2, oznacza to że model nie radzi sobie najlepiej z klasyfikacją jednak wyniki są lepsze niż losowe. 
W tym przypadku model nie miał być jednak użyty do predykcji, a bardziej do analizy, które czynniki okażą się istotne, dlatego wyniki i tak uznaję za ciekawe.